In [ ]:
%pip install dotenv
%pip install datasets
%pip install scikit-learn
%pip install -r requirements.txt
%load_ext autoreload
%autoreload 2


In [ ]:
import sys

import torch

sys.path.append(".")


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


In [ ]:
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("CPU cores:", os.cpu_count())


In [ ]:
from huggingface_hub import login
import os
from dotenv import load_dotenv

load_dotenv()

# Use your token to log in
login(token=os.getenv("hf_token"))


# BASIL: lexical and informational bias detection

Fine-tunes the pretrained checkpoints and the baselines on
[BASIL](https://github.com/launchnlp/BASIL) (EMNLP 2019), read from the local checkout at
`BASIL/`. Two binary sentence-classification tasks over the same 7,984 body sentences:

- **lexical** — does the sentence contain a lexical-bias span? 449 positives (5.6%).
- **informational** — does it contain an informational-bias span? 1,218 positives (15.3%).

Each sentence goes in alone, as the paper's footnote 4 specifies: "we treat sentences as
passages, rather than using text of fixed length". Both tasks ride the `bias_*` keys and
`bias_head`, sized to 2 classes rather than the 3 that AllSides, MITweet ideology and SemEval
use.

**The split** reproduces the paper's: 6,819 train / 758 validation / 400 test, stratified on
the task's label. Stratification is what makes the positives land on BASIL's published
per-split counts — 1,043/116/62 informational and 383/42/23 lexical.

**Ten folds.** The paper reports the mean of 10-fold cross validation, but its split sizes are
not a disjoint partition (a tenth of 7,984 is 798, not 400), so a fold here is a fresh
stratified split seeded by the fold index.

**Class weighting.** `BasilTrainer` computes a weighted cross-entropy itself for *every* model,
so a `MultiTaskRoberta` and an HF baseline optimise the identical objective. At 5.6% positive
an unweighted loss collapses to predicting the negative class. This is an addition to the
paper, which does not say how it handled the imbalance.

In [ ]:
import glob
from pathlib import Path

from huggingface_hub import snapshot_download

from config import load_run_config
from finetuning import aggregate as agg
from finetuning.experiments import run_basil_experiment, ExperimentConfig
from finetuning.basil import NUM_FOLDS, TASKS, BasilConfig, variant_name
from finetuning.models import BERT, BART, ROBERTA, POLITICS, IDEOLOGY_CLASSIFIER


In [ ]:
ideology_dir = snapshot_download(repo_id=IDEOLOGY_CLASSIFIER)
ideology_pt = glob.glob(f"{ideology_dir}/*.pt")[0]
print(f"Ideology classifier .pt: {ideology_pt}")


In [ ]:
def find_tlp_checkpoints(config_glob="run_configs/tlp_*.yaml"):
    """Locate the checkpoint each tlp_* pretraining run left behind.
    """
    checkpoints = []
    for config_path in sorted(glob.glob(config_glob)):
        label = Path(config_path).stem
        output_dir = Path(load_run_config(config_path).output_dir)
        epochs = sorted(
            output_dir.glob("epoch-*.pt"),
            key=lambda p: int(p.stem.split("-")[1]),
        )
        if not epochs:
            print(f"  SKIP {label}: no epoch-*.pt under {output_dir}")
            continue
        print(f"  {label}: {epochs[-1]}")
        checkpoints.append((str(epochs[-1]), label))
    return checkpoints


print("tlp checkpoints:")
TLP_CHECKPOINTS = find_tlp_checkpoints()
print(f"\nfound {len(TLP_CHECKPOINTS)} of 4")


In [ ]:
BASELINES = [
    (BERT, "bert"),
    (BART, "bart"),
    (ROBERTA, "roberta"),
    (POLITICS, "politics"),
    (ideology_pt, "ideology"),
]

BASIL_ROOT = "results_basil"


def run_all_models(models, basil_config, seed, root=BASIL_ROOT):
    """Fine-tune each of `models` on one BASIL task and fold, under one seed.
    """
    # One directory per (task, fold), so `agg.discover_results` reads a single comparison.
    loc = f"{root}/{variant_name(basil_config)}/fold_{basil_config.fold}/seed_{seed}"
    os.makedirs(loc, exist_ok=True)
    results = {}
    for model_ref, model_name in models:
        exp = ExperimentConfig(patience=2, num_epochs=10, save_model=False, seed=seed)
        print(f"\n{'='*60}")
        print(f"Model: {model_name}  |  Task: {basil_config.task}  "
              f"|  Fold: {basil_config.fold}  |  Seed: {seed}")
        print('='*60)
        results[model_name] = run_basil_experiment(
            model=model_ref,
            loc=loc,
            basil_config=basil_config,
            experiment_config=exp,
            model_name=model_name,
        )
    return results


In [ ]:
MODELS = BASELINES + TLP_CHECKPOINTS

# The paper's number is the mean over the ten folds, so one seed per fold already reproduces
# it. Each extra seed multiplies by ten: 9 models x 2 tasks x 10 folds x 5 seeds = 900 runs.
SEEDS = [42]

print(f"{len(MODELS)} models x {len(TASKS)} tasks x {NUM_FOLDS} folds x {len(SEEDS)} seeds "
      f"= {len(MODELS) * len(TASKS) * NUM_FOLDS * len(SEEDS)} runs")


## Lexical bias

The harder of the two and the rarer: 449 of 7,984 sentences, 22 of any given 400-sentence test
split. BASIL's fine-tuned BERT reaches P/R/F1 29.13/38.57/31.49 here.

In [ ]:
for fold in range(NUM_FOLDS):
    for seed in SEEDS:
        print(f"\n{'#'*60}\nlexical, fold {fold}, seed {seed}\n{'#'*60}")
        run_all_models(
            models=MODELS,
            basil_config=BasilConfig(task="lexical", fold=fold),
            seed=seed,
        )


## Informational bias

Three times as common as lexical bias and, per the paper, better captured by sentence-level
context: BASIL's fine-tuned BERT reaches P/R/F1 43.87/42.91/43.27.

In [ ]:
for fold in range(NUM_FOLDS):
    for seed in SEEDS:
        print(f"\n{'#'*60}\ninformational, fold {fold}, seed {seed}\n{'#'*60}")
        run_all_models(
            models=MODELS,
            basil_config=BasilConfig(task="informational", fold=fold),
            seed=seed,
        )


## Cross-fold analysis

Everything below reads the per-fold metrics JSONs off disk — no model, no GPU. The runs above
do not need to have happened in this session.

`f1_positive` is the paper's reported F1 and what the best epoch was selected on; `f1_macro`
is over both classes, and on a 94%-negative task it stays high whatever the model does, so
read the two together.

In [ ]:
TASK = "lexical"

# One comparison per fold; `agg` needs no changes because the leaf is still `seed_N`.
per_fold = agg.discover_results(f"{BASIL_ROOT}/{TASK}/fold_0")
print(f"{len(per_fold)} runs in fold 0")
agg.coverage(per_fold)


In [ ]:
BASIL_METRICS = ["f1_positive", "precision_positive", "recall_positive", "f1_macro", "accuracy"]

agg.summary_table(per_fold, metrics=BASIL_METRICS)


In [ ]:
# The paper's read-out: each model's mean over the ten folds, and the spread across them.
# BASIL reports fold standard deviations of 3.36 to 12.44, so expect a wide band.
import json

import pandas as pd

rows = []
for task in TASKS:
    for fold in range(NUM_FOLDS):
        runs = agg.discover_results(f"{BASIL_ROOT}/{task}/fold_{fold}")
        if not runs:
            continue
        for run in runs:
            payload = json.load(open(run.path))
            rows.append({
                "task": task, "fold": fold, "model": run.model, "seed": run.seed,
                **{metric: payload.get(metric) for metric in BASIL_METRICS},
            })

frame = pd.DataFrame(rows)
# Mean over seeds within a fold first, so a model run at more seeds is not weighted higher.
by_fold = frame.groupby(["task", "model", "fold"])[BASIL_METRICS].mean()
by_fold.groupby(["task", "model"]).agg(["mean", "std", "count"]).round(2)


In [ ]:
# Does a model only work on one outlet? Per-source positive-class F1, pooled over folds.
SOURCE_METRICS = ["f1_source_fox", "f1_source_hpo", "f1_source_nyt"]

rows = []
for task in TASKS:
    for fold in range(NUM_FOLDS):
        for run in agg.discover_results(f"{BASIL_ROOT}/{task}/fold_{fold}"):
            payload = json.load(open(run.path))
            rows.append({"task": task, "model": run.model,
                         **{m: payload.get(m) for m in SOURCE_METRICS}})

pd.DataFrame(rows).groupby(["task", "model"])[SOURCE_METRICS].mean().round(2)


In [ ]:
# Ensemble and error read-outs for one fold: how much the seeds agree, and which way the
# errors go. Only meaningful when SEEDS has more than one entry.
agg.print_reports(agg.discover_results(f"{BASIL_ROOT}/{TASK}/fold_0"))
